# 02 - Time-Series Cleaning, Missing-Value Imputation & Resampling
This notebook structures timestamps into a continuous temporal index, imputes missing records using time-weighted interpolation, and resamples to a uniform 15-minute / 1-hour interval.


In [ ]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv('../data/raw/household_power_consumption.txt', sep=';', na_values=['?'], nrows=20000)
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp').set_index('timestamp')
df = df.drop(columns=['Date', 'Time'])
df.head()


### Forward-Fill & Time-Interpolation
We impute short gaps via forward-fill and linear interpolation so the physical continuity of energy consumption is preserved.


In [ ]:
num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df[num_cols] = df[num_cols].ffill().bfill()
print("Missing values after imputation:
", df.isnull().sum())


### Physics-Aware Resampling (15-Minute / 1-Hour)
Power (kW) uses mean aggregation. Energy consumed in kWh over an hour equals mean(kW) * 1h.


In [ ]:
df_hourly = df[num_cols].resample('1h').mean()
df_hourly['energy_consumption_kwh'] = df_hourly['Global_active_power'] * 1.0
df_hourly.head()


### Summary
Cleaned, uniform time series resampled and ready for feature engineering.
